In [1]:
# ! pip3 install tavily-python

In [2]:
import getpass
from openai import OpenAI

# Securely prompt for the API key
api_key = getpass.getpass("Enter your OpenAI API Key: ")

In [3]:
import json
from typing import Callable


def get_fn_signature(fn: Callable) -> dict:
    """
    Generates the signature for a given function.

    Args:
        fn (Callable): The function whose signature needs to be extracted.

    Returns:
        dict: A dictionary containing the function's name, description,
              and parameter types.
    """
    fn_signature: dict = {
        "name": fn.__name__,
        "description": fn.__doc__,
        "parameters": {"properties": {}},
    }
    schema = {
        k: {"type": v.__name__} for k, v in fn.__annotations__.items() if k != "return"
    }
    fn_signature["parameters"]["properties"] = schema
    return fn_signature


def validate_arguments(tool_call: dict, tool_signature: dict) -> dict:
    """
    Validates and converts arguments in the input dictionary to match the expected types.

    Args:
        tool_call (dict): A dictionary containing the arguments passed to the tool.
        tool_signature (dict): The expected function signature and parameter types.

    Returns:
        dict: The tool call dictionary with the arguments converted to the correct types if necessary.
    """
    properties = tool_signature["parameters"]["properties"]

    # TODO: This is overly simplified but enough for simple Tools.
    type_mapping = {
        "int": int,
        "str": str,
        "bool": bool,
        "float": float,
        "list": list,
    }

    for arg_name, arg_value in tool_call["arguments"].items():
        expected_type = properties[arg_name].get("type")

        if not isinstance(arg_value, type_mapping[expected_type]):
            tool_call["arguments"][arg_name] = type_mapping[expected_type](arg_value)

    return tool_call


class Tool:
    """
    A class representing a tool that wraps a callable and its signature.

    Attributes:
        name (str): The name of the tool (function).
        fn (Callable): The function that the tool represents.
        fn_signature (str): JSON string representation of the function's signature.
    """

    def __init__(self, name: str, fn: Callable, fn_signature: str):
        self.name = name
        self.fn = fn
        self.fn_signature = fn_signature

    def __str__(self):
        return self.fn_signature

    def run(self, **kwargs):
        """
        Executes the tool (function) with provided arguments.

        Args:
            **kwargs: Keyword arguments passed to the function.

        Returns:
            The result of the function call.
        """
        return self.fn(**kwargs)


def tool(fn: Callable):
    """
    A decorator that wraps a function into a Tool object.

    Args:
        fn (Callable): The function to be wrapped.

    Returns:
        Tool: A Tool object containing the function, its name, and its signature.
    """

    def wrapper():
        fn_signature = get_fn_signature(fn)
        return Tool(
            name=fn_signature.get("name"), fn=fn, fn_signature=json.dumps(fn_signature)
        )

    return wrapper()

### Project : Tool Function to fetch search from Tavily

In [4]:
import json
import importlib

try:
    tavily = importlib.import_module('tavily')
except Exception:
    tavily = None

@tool
def tavily_search(questions: list[str], max_results: int = 3):
    '''
    Search queries using the Tavily library and return title+content.

    Args:
        questions (list[str]): list of search queries.
        max_results (int): max results to return per query.

    Returns:
        JSON string: list of dicts with keys query and results (each result has title and content).
    '''
    if tavily is None:
        raise ImportError('tavily is not installed. Install it with: pip install tavily')

    all_queries = []

    def _extract(item):
        if item is None:
            return {'title': '', 'content': ''}
        if isinstance(item, dict):
            title = item.get('title') or item.get('name') or item.get('headline') or ''
            content = item.get('content') or item.get('raw_content') or item.get('snippet') or item.get('text') or ''
            return {'title': title, 'content': content}
        title = getattr(item, 'title', None) or getattr(item, 'name', '')
        content = getattr(item, 'content', None) or getattr(item, 'raw_content', None) or getattr(item, 'snippet', None) or getattr(item, 'text', None) or ''
        return {'title': title or '', 'content': content or ''}

    search_fn = None
    if hasattr(tavily, 'search') and callable(getattr(tavily, 'search')):
        search_fn = tavily.search
    else:
        for name in ('Tavily', 'TavilyClient', 'Client'):
            cls = getattr(tavily, name, None)
            if cls is not None:
                try:
                    client = cls()
                    if hasattr(client, 'search') and callable(getattr(client, 'search')):
                        search_fn = client.search
                        break
                except Exception:
                    continue

    if search_fn is None:
        raise RuntimeError('Could not find a usable search function in tavily module.')

    for q in questions:
        try:
            try:
                raw = search_fn(q, k=max_results)
            except TypeError:
                try:
                    raw = search_fn(q, max_results)
                except TypeError:
                    raw = search_fn(q)

            q_results = []

            # Normalize different response shapes (tavily may return a dict with 'results')
            items = []
            if isinstance(raw, dict):
                if 'results' in raw and isinstance(raw['results'], (list, tuple)):
                    items = raw['results'][:max_results]
                elif 'results' in raw and isinstance(raw['results'], dict):
                    items = [raw['results']]
                elif 'data' in raw and isinstance(raw['data'], (list, tuple)):
                    items = raw['data'][:max_results]
                else:
                    items = [raw]
            elif isinstance(raw, (list, tuple)):
                items = list(raw)[:max_results]
            else:
                items = [raw]

            for item in items:
                q_results.append(_extract(item))

            all_queries.append({'query': q, 'results': q_results})
        except Exception as e:
            all_queries.append({'query': q, 'results': [], 'error': str(e)})

    return json.dumps(all_queries)

In [5]:
json.loads(tavily_search.fn_signature)

{'name': 'tavily_search',
 'description': '\nSearch queries using the Tavily library and return title+content.\n\nArgs:\n    questions (list[str]): list of search queries.\n    max_results (int): max results to return per query.\n\nReturns:\n    JSON string: list of dicts with keys query and results (each result has title and content).\n',
 'parameters': {'properties': {'questions': {'type': 'list'},
   'max_results': {'type': 'int'}}}}

### Building the Tool-Using Agent

In [6]:
# @title
"""
This is a collection of helper functions and methods we are going to use in
the Agent implementation. You don't need to know the specific implementation
of these to follow the Agent code. But, if you are curious, feel free to check
them out.
"""

import re
import time

from colorama import Fore
from colorama import Style

from dataclasses import dataclass


def completions_create(client, messages: list, model: str) -> str:
    """
    Sends a request to the client's `completions.create` method to interact with the language model.

    Args:
        client (OpenAI): The OpenAI client object
        messages (list[dict]): A list of message objects containing chat history for the model.
        model (str): The model to use for generating tool calls and responses.

    Returns:
        str: The content of the model's response.
    """
    response = client.chat.completions.create(messages=messages, model=model,max_tokens=2000)
    return str(response.choices[0].message.content)


def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    """
    Builds a structured prompt that includes the role and content.

    Args:
        prompt (str): The actual content of the prompt.
        role (str): The role of the speaker (e.g., user, assistant).

    Returns:
        dict: A dictionary representing the structured prompt.
    """
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}

def update_chat_history(history: list, msg: str, role: str):
    """
    Updates the chat history by appending the latest response.

    Args:
        history (list): The list representing the current chat history.
        msg (str): The message to append.
        role (str): The role type (e.g. 'user', 'assistant', 'system')
    """
    history.append(build_prompt_structure(prompt=msg, role=role))


class ChatHistory(list):
    def __init__(self, messages: list | None = None, total_length: int = -1):
        """Initialise the queue with a fixed total length.

        Args:
            messages (list | None): A list of initial messages
            total_length (int): The maximum number of messages the chat history can hold.
        """
        if messages is None:
            messages = []

        super().__init__(messages)
        self.total_length = total_length

    def append(self, msg: str):
        """Add a message to the queue.

        Args:
            msg (str): The message to be added to the queue
        """
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)



class FixedFirstChatHistory(ChatHistory):
    def __init__(self, messages: list | None = None, total_length: int = -1):
        """Initialise the queue with a fixed total length.

        Args:
            messages (list | None): A list of initial messages
            total_length (int): The maximum number of messages the chat history can hold.
        """
        super().__init__(messages, total_length)

    def append(self, msg: str):
        """Add a message to the queue. The first messaage will always stay fixed.

        Args:
            msg (str): The message to be added to the queue
        """
        if len(self) == self.total_length:
            self.pop(1)
        super().append(msg)

def fancy_print(message: str) -> None:
    """
    Displays a fancy print message.

    Args:
        message (str): The message to display.
    """
    print(Style.BRIGHT + Fore.CYAN + f"\n{'=' * 50}")
    print(Fore.MAGENTA + f"{message}")
    print(Style.BRIGHT + Fore.CYAN + f"{'=' * 50}\n")
    time.sleep(0.5)


def fancy_step_tracker(step: int, total_steps: int) -> None:
    """
    Displays a fancy step tracker for each iteration of the generation-reflection loop.

    Args:
        step (int): The current step in the loop.
        total_steps (int): The total number of steps in the loop.
    """
    fancy_print(f"STEP {step + 1}/{total_steps}")


@dataclass
class TagContentResult:
    """
    A data class to represent the result of extracting tag content.

    Attributes:
        content (List[str]): A list of strings containing the content found between the specified tags.
        found (bool): A flag indicating whether any content was found for the given tag.
    """

    content: list[str]
    found: bool


def extract_tag_content(text: str, tag: str) -> TagContentResult:
    """
    Extracts all content enclosed by specified tags (e.g., <thought>, <response>, etc.).

    Parameters:
        text (str): The input string containing multiple potential tags.
        tag (str): The name of the tag to search for (e.g., 'thought', 'response').

    Returns:
        dict: A dictionary with the following keys:
            - 'content' (list): A list of strings containing the content found between the specified tags.
            - 'found' (bool): A flag indicating whether any content was found for the given tag.
    """
    # Build the regex pattern dynamically to find multiple occurrences of the tag
    tag_pattern = rf"<{tag}>(.*?)</{tag}>"

    # Use findall to capture all content between the specified tag
    matched_contents = re.findall(tag_pattern, text, re.DOTALL)

    # Return the dataclass instance with the result
    return TagContentResult(
        content=[content.strip() for content in matched_contents],
        found=bool(matched_contents),
    )


### Project : System Prompt

In [7]:
import json
import re

from colorama import Fore
from openai import OpenAI


TOOL_SYSTEM_PROMPT = """
You are a function-calling AI model. You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Do not make assumptions about what values to plug into functions; pay special attention to the properties 'types' and use those types as in a Python dict.
For each function call return a JSON object with the function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools:

<tools>
%s
</tools>

If the user asks about something related to the tool, always provide the tool call. If a tool can provide real-time or external information relevant to the user request, prefer calling that tool rather than claiming you cannot access external information.

Additional behavior for question-generation use-cases:
When the user asks you to generate research or exploration questions for a topic (for example: 'Generate questions about Climate Change'), do the following:
- Produce 5–6 well-structured, concise, and varied questions that each cover a distinct aspect of the topic (for example: definition/overview, causes/factors, historical trends/data, impacts/consequences, mitigation/solutions/policies, examples/case studies).
- Present the questions as a numbered list (1–5 or 1–6).
- Make each question self-contained and unambiguous; prefer open-ended prompts that invite explanation or investigation.
- Avoid overlapping content between questions; each should focus on a unique angle.

Example (topic = Climate Change):
1. What are the main causes of climate change?
2. How has global temperature changed over the past century?
3. What are the primary impacts of climate change on ecosystems and human societies?
4. Which policies and international agreements aim to mitigate climate change, and how effective are they?
5. What technological and behavioral solutions are available to reduce greenhouse gas emissions?
6. Can you provide examples of countries or regions that have successfully implemented adaptation strategies?

Operational rules:
- When producing a tool call, ensure all arguments match the function signature types exactly.
- If the user explicitly requests suggested search queries or to run searches, generate the questions first and then, if appropriate, return a tool call to perform the searches.
"""


class ToolAgent:
    """
    The ToolAgent class represents an agent that can interact with a language model and use tools
    to assist with user queries. It generates function calls based on user input, validates arguments,
    and runs the respective tools.

    Attributes:
        tools (Tool | list[Tool]): A list of tools available to the agent.
        model (str): The model to be used for generating tool calls and responses.
        client (OpenAI): The OpenAI client used to interact with the language model.
        tools_dict (dict): A dictionary mapping tool names to their corresponding Tool objects.
    """

    def __init__(
        self,
        tools: Tool | list[Tool],
        model: str = "gpt-4o-mini",
    ) -> None:
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.tools = tools if isinstance(tools, list) else [tools]
        self.tools_dict = {tool.name: tool for tool in self.tools}

    def add_tool_signatures(self) -> str:
        """
        Collects the function signatures of all available tools.

        Returns:
            str: A concatenated string of all tool function signatures in JSON format.
        """
        return "".join([tool.fn_signature for tool in self.tools])

    def process_tool_calls(self, tool_calls_content: list) -> dict:
        """
        Processes each tool call, validates arguments, executes the tools, and collects results.

        Args:
            tool_calls_content (list): List of strings, each representing a tool call in JSON format.

        Returns:
            dict: A dictionary where the keys are tool call IDs and values are the results from the tools.
        """
        observations = {}
        next_id = 1
        for tool_call_str in tool_calls_content:
            try:
                tool_call = json.loads(tool_call_str)
            except Exception as e:
                print(Fore.RED + f"\nFailed to parse tool_call JSON: {e}")
                continue

            # Ensure an 'id' exists; assign a local monotonically increasing id if missing
            if "id" not in tool_call:
                tool_call["id"] = next_id
            try:
                cid = tool_call["id"]
            except Exception:
                cid = next_id

            next_id += 1

            tool_name = tool_call.get("name")
            tool = self.tools_dict.get(tool_name)
            if tool is None:
                print(Fore.RED + f"\nUnknown tool: {tool_name}")
                observations[cid] = {"error": f"unknown tool: {tool_name}"}
                continue

            print(Fore.GREEN + f"\nUsing Tool: {tool_name}")

            # Validate and execute the tool call
            validated_tool_call = validate_arguments(
                tool_call, json.loads(tool.fn_signature)
            )
            print(Fore.GREEN + f"\nTool call dict: \n+{validated_tool_call}")

            try:
                result = tool.run(**validated_tool_call["arguments"])
            except Exception as e:
                result = {"error": str(e)}
            print(Fore.GREEN + f"\nTool result: \n+{result}")

            observations[cid] = result

        return observations

    def run(
        self,
        user_msg: str,
    ) -> str:
        """
        Handles the full process of interacting with the language model and executing a tool based on user input.

        Args:
            user_msg (str): The user's message that prompts the tool agent to act.

        Returns:
            str: The final output after executing the tool and generating a response from the model.
        """
        user_prompt = build_prompt_structure(prompt=user_msg, role="user")

        tool_chat_history = ChatHistory(
            [
                build_prompt_structure(
                    prompt=TOOL_SYSTEM_PROMPT % self.add_tool_signatures(),
                    role="system",
                ),
                user_prompt,
            ]
        )
        agent_chat_history = ChatHistory([user_prompt])

        tool_call_response = completions_create(
            self.client, messages=tool_chat_history, model=self.model
        )
        
        print(Fore.BLUE + f"\nModel response with potential tool calls:\n{tool_call_response}")
        
        tool_calls = extract_tag_content(str(tool_call_response), "tool_call")
        
        if tool_calls.found:
            observations = self.process_tool_calls(tool_calls.content)
            update_chat_history(
                agent_chat_history, f'f"Observation: {observations}"', "user"
            )

        return completions_create(self.client, agent_chat_history, self.model)


In [8]:
tool_agent = ToolAgent(tools=[tavily_search])

In [9]:
output = tool_agent.run(user_msg="Challenges in Indian Education System")


Model response with potential tool calls:
Here are some well-structured questions regarding the challenges in the Indian education system:

1. What are the primary structural challenges faced by the Indian education system, including issues related to accessibility and infrastructure?
2. How does the quality of teacher training and professional development impact student outcomes in India?
3. In what ways does socioeconomic status influence educational opportunities and achievement among Indian students?
4. What is the role of outdated curriculum and assessment methods in hindering educational effectiveness in India?
5. How do regional disparities affect the overall education system in India, particularly between urban and rural areas?
6. What measures are being taken to address the challenges of dropout rates and ensure retention in schools across India? 

Now, I will proceed to search for more information related to these challenges in the Indian education system.


In [10]:
print(Fore.CYAN + output)

The Indian education system faces a variety of challenges that impact its effectiveness and accessibility. Here are some of the key challenges:

1. **Quality of Education**: There is a significant disparity in the quality of education across urban and rural areas. Many rural schools lack qualified teachers, proper infrastructure, and teaching materials.

2. **High Dropout Rates**: Students often drop out of school due to economic pressures, the need to support their families, or the irrelevance of the curriculum to their lives and aspirations. This is particularly prevalent in lower socio-economic groups.

3. **Exam-Oriented Culture**: The emphasis on rote learning and high-stakes examinations can stifle creativity and critical thinking. Students often focus on passing exams rather than gaining a deeper understanding of the subjects.

4. **Access and Inclusivity**: Although there have been initiatives to increase access to education, many marginalized communities still face barriers. I